# Replikasi Terbatas — Nasution, Onan, Murakami, Monika & Hanafiah (2025)
## *Benchmarking Open-Source Large Language Models for Sentiment and Emotion Classification in Indonesian Tweets*

**Jurnal:** IEEE Access Vol. 13 · DOI `10.1109/ACCESS.2025.3574629`

---

## Hasil audit: datanya publik, komputasinya tidak

| Komponen | Status |
|---|---|
| Dataset sentimen (IndoNLU SmSA, 12.760) | ✅ **publik** |
| Dataset emosi (IndoNLU EmoT, 4.401) | ✅ **publik** |
| Prompt zero-shot | ✅ **dikutip verbatim di paper** |
| Kode | ❌ tidak ada |
| 22 LLM + ChatGPT-4 | ❌ butuh 4× RTX A6000 (49 GB) |
| Pernyataan Data Availability | ❌ **tidak ada** |

### Bagian yang benar-benar bisa dikerjakan

Dataset dan prompt-nya lengkap, jadi zero-shot benchmark **bisa** direplikasi — untuk model kecil. Yang tidak bisa: 70B (LLaMA 3.1/3.3, DeepSeek-R1 70B) dan ChatGPT-4.

Hardware paper: **4× NVIDIA RTX A6000 49 GB**, CUDA 12.6, lewat Ollama untuk model terkuantisasi. Colab T4 16 GB hanya sanggup model ≤7B dalam 4-bit.

### Prompt paper (dikutip apa adanya)

**Sentimen:**
> `Tweet text: [tweet]. What is the sentiment of this tweet? Respond with one of: Positive, Negative, or Neutral.`

**Emosi:**
> `Tweet text: [tweet]. Identify which of the following emotions are expressed (you may choose multiple if applicable): Love, Happiness, Sadness, Anger, Fear. Respond with the emotion labels.`

### Angka acuan paper (macro-F1)

| Model | Sentimen | Catatan |
|---|---|---|
| ChatGPT-4 | 0,845 | butuh API berbayar |
| Human annotators | 0,846 | batas atas |
| LLaMA3.1_70B / LLaMA3.3_70B | terbaik di antara open-source | butuh ~140 GB |
| Gemma2_27B | kompetitif | butuh ~54 GB |

> **Catatan keanehan paper:** mereka menulis LLaMA3.3 sebagai *"representing hypothetical successive versions of LLaMA (we assume these are improved variants)"*. Kata "hypothetical" dan "we assume" di bagian model selection itu tidak lazim untuk benchmark — artinya penulis sendiri tidak sepenuhnya yakin model apa yang mereka jalankan.


## 1. Setup

In [ ]:
!pip install -q "torch>=2.1" "transformers>=4.45" "accelerate>=0.30" bitsandbytes pandas scikit-learn

import os, re, json, random, time, urllib.request
import numpy as np, pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import f1_score, classification_report

SEED = 42          # [ASUMSI] paper tidak menyebut seed
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def show(d):
    try:
        from IPython.display import display as _d; _d(d)
    except Exception:
        print(d.to_string())
print('device:', DEVICE)

device: cuda


## 2. Unduh Kedua Dataset IndoNLU

SmSA: 11.000 train / 1.260 valid / 500 test = **12.760** (TSV, kolom `text`+`label`).
EmoT: 3.521 / 440 / 440 = **4.401** (CSV, kolom `tweet`+`label`).

In [ ]:
SMSA = 'https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/smsa_doc-sentiment-prosa/'
EMOT = 'https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/emot_emotion-twitter/'
os.makedirs('indonlu', exist_ok=True)

def fetch(url, path):
    if not os.path.exists(path):
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=180) as r, open(path, 'wb') as f:
            f.write(r.read())
    return path

smsa = {}
for sp in ('train', 'valid', 'test'):
    p = fetch(f'{SMSA}{sp}_preprocess.tsv', f'indonlu/smsa_{sp}.tsv')
    d = pd.read_csv(p, sep='\t', header=None, names=['text', 'label'])
    smsa[sp] = d.dropna().reset_index(drop=True)
    print(f'smsa/{sp}: {len(d)}')

emot = {}
for sp in ('train', 'valid', 'test'):
    p = fetch(f'{EMOT}{sp}_preprocess.csv', f'indonlu/emot_{sp}.csv')
    d = pd.read_csv(p)
    emot[sp] = d.dropna().reset_index(drop=True)
    print(f'emot/{sp}: {len(d)}')

smsa_all = pd.concat(smsa.values()); emot_all = pd.concat(emot.values())
print(f'\nSmSA total {len(smsa_all)} | EmoT total {len(emot_all)}')
assert len(smsa_all) == 12760, f'SmSA harus 12.760, dapat {len(smsa_all)}'
assert len(emot_all) == 4401, f'EmoT harus 4.401, dapat {len(emot_all)}'
print('[OK] kedua dataset cocok persis dengan angka paper (12.760 / 4.401)')

smsa/train: 11000
smsa/valid: 1260
smsa/test: 500
emot/train: 3521
emot/valid: 440
emot/test: 440

SmSA total 12760 | EmoT total 4401
[OK] kedua dataset cocok persis dengan angka paper (12.760 / 4.401)


## 3. Sanity Check

In [ ]:
print('--- SmSA (sentimen) ---')
print(smsa_all['label'].value_counts().to_dict())
print('contoh:', smsa_all['text'].iloc[0][:90])

print('\n--- EmoT (emosi) ---')
print(emot_all['label'].value_counts().to_dict())
print('contoh:', emot_all['tweet'].iloc[0][:90])

SENT_LABELS = ['positive', 'negative', 'neutral']
EMO_LABELS  = ['love', 'happy', 'sadness', 'anger', 'fear']

assert set(smsa_all['label'].unique()) <= set(SENT_LABELS)
assert set(emot_all['label'].unique()) <= set(EMO_LABELS)
print('\n[OK] label sesuai 3 kelas sentimen & 5 kelas emosi')

--- SmSA (sentimen) ---
{'positive': 7359, 'negative': 4034, 'neutral': 1367}
contoh: warung ini dimiliki oleh pengusaha pabrik tahu yang sudah puluhan tahun terkenal membuat t

--- EmoT (emosi) ---
{'anger': 1101, 'happy': 1017, 'sadness': 997, 'fear': 649, 'love': 637}
contoh: Ini adalah hal yang paling membahagiakan saat biasku foto bersama ELF #ReturnOfTheLittlePr

[OK] label sesuai 3 kelas sentimen & 5 kelas emosi


## 4. Zero-shot Prompt (persis paper)

`[ASUMSI]` model dipanggil dengan `temperature=0` dan `max_new_tokens=8` supaya jawabannya cuma label. Paper tidak menyebut setelan sampling.

In [ ]:
SENT_PROMPT = ('Tweet text: {text}. What is the sentiment of this tweet? '
               'Respond with one of: Positive, Negative, or Neutral.')
EMO_PROMPT  = ('Tweet text: {text}. Identify which of the following emotions are expressed '
               '(you may choose multiple if applicable): Love, Happiness, Sadness, Anger, Fear. '
               'Respond with the emotion labels.')

def parse_sent(raw):
    t = raw.lower()
    for lab in ('positive', 'negative', 'neutral'):
        if lab in t:
            return lab
    return 'neutral'          # [ASUMSI] fallback kalau model tidak menjawab label

def parse_emo(raw):
    t = raw.lower()
    for lab in EMO_LABELS:
        if lab in t:
            return lab
    if 'happiness' in t or 'joy' in t:
        return 'happy'
    return 'happy'            # [ASUMSI] fallback

print('prompt sentimen:'); print(' ', SENT_PROMPT)
print('prompt emosi  :'); print(' ', EMO_PROMPT)

# uji parser tanpa model
assert parse_sent('Positive') == 'positive'
assert parse_sent('the answer is Negative.') == 'negative'
assert parse_emo('Happiness') == 'happiness' or parse_emo('Happiness') == 'happy'
print('\n[OK] parser label berfungsi')

prompt sentimen:
  Tweet text: {text}. What is the sentiment of this tweet? Respond with one of: Positive, Negative, or Neutral.
prompt emosi  :
  Tweet text: {text}. Identify which of the following emotions are expressed (you may choose multiple if applicable): Love, Happiness, Sadness, Anger, Fear. Respond with the emotion labels.

[OK] parser label berfungsi


## 5. Jalankan Zero-shot pada Model Kecil

Model default: `Qwen/Qwen2.5-3B-Instruct` (tanpa gating, muat di T4 dengan 4-bit).

Paper memakai 22 model termasuk 70B. Di sini hanya model ≤9B yang realistis.

> ⚠️ **Belum diuji** — butuh GPU. Zero-shot pada 500 tweet uji sentimen ≈ 5–10 menit untuk model 3B.

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'    # ganti sesuai selera
N_SAMPLES = 500                          # paper memakai seluruh test set
RUN_INFERENCE = True

if RUN_INFERENCE:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    lm = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                              device_map='auto').eval()

    def ask(prompt):
        msgs = [{'role': 'user', 'content': prompt}]
        # Tambahkan return_dict=False atau langsung ambil tensor input_ids-nya
        ids = tok.apply_chat_template(msgs, return_tensors='pt',
                                      add_generation_prompt=True, return_dict=False).to(lm.device)
        with torch.no_grad():
            out = lm.generate(ids, max_new_tokens=8, do_sample=False,
                              pad_token_id=tok.eos_token_id)
        return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

    # --- sentimen ---
    t0 = time.time()
    sub = smsa['test'].head(N_SAMPLES)
    preds = [parse_sent(ask(SENT_PROMPT.format(text=t))) for t in sub['text']]
    gold = sub['label'].tolist()
    print(f'sentimen: {len(sub)} sampel, {time.time()-t0:.0f}s')
    print(classification_report(gold, preds, zero_division=0, digits=4))
    sent_f1 = f1_score(gold, preds, average='macro')

    # --- emosi ---
    t0 = time.time()
    sub_e = emot['test']
    preds_e = [parse_emo(ask(EMO_PROMPT.format(text=t))) for t in sub_e['tweet']]
    gold_e = sub_e['label'].tolist()
    print(f'\nemosi: {len(sub_e)} sampel, {time.time()-t0:.0f}s')
    print(classification_report(gold_e, preds_e, zero_division=0, digits=4))
    emo_f1 = f1_score(gold_e, preds_e, average='macro')

    print(f'\nmacro-F1 sentimen: {sent_f1:.4f}')
    print(f'macro-F1 emosi  : {emo_f1:.4f}')
else:
    print('RUN_INFERENCE=False — set True untuk menjalankan (butuh GPU).')

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

sentimen: 500 sampel, 252s
              precision    recall  f1-score   support

    negative     0.7656    0.7206    0.7424       204
     neutral     0.3144    0.8182    0.4543        88
    positive     0.9747    0.3702    0.5366       208

    accuracy                         0.5920       500
   macro avg     0.6849    0.6363    0.5778       500
weighted avg     0.7732    0.5920    0.6061       500


emosi: 440 sampel, 208s
              precision    recall  f1-score   support

       anger     0.5543    0.4636    0.5050       110
        fear     0.7000    0.3231    0.4421        65
       happy     0.5000    0.3069    0.3804       101
        love     0.4359    0.5312    0.4789        64
     sadness     0.4045    0.7200    0.5180       100

    accuracy                         0.4750       440
   macro avg     0.5189    0.4690    0.4649       440
weighted avg     0.5121    0.4750    0.4662       440


macro-F1 sentimen: 0.5778
macro-F1 emosi  : 0.4649


## 6. Tabel Perbandingan

In [ ]:
PAPER = [
    {'Model': 'ChatGPT-4',            'Macro-F1': 0.845, 'Bisa di Colab?': 'tidak (API berbayar)'},
    {'Model': 'Human annotators',     'Macro-F1': 0.846, 'Bisa di Colab?': 'ya (label asli)'},
    {'Model': 'LLaMA3.3_70B',         'Macro-F1': 0.846, 'Bisa di Colab?': 'tidak (~140 GB)'},
    {'Model': 'LLaMA3.1_70B',         'Macro-F1': None,  'Bisa di Colab?': 'tidak (~140 GB)'},
    {'Model': 'Gemma2_27B',           'Macro-F1': None,  'Bisa di Colab?': 'tidak (~54 GB)'},
]
rows = list(PAPER)
if RUN_INFERENCE and 'sent_f1' in dir():
    rows.append({'Model': MODEL_ID, 'Macro-F1': round(sent_f1, 4), 'Bisa di Colab?': 'ya'})
show(pd.DataFrame(rows))
print('\nKolom Macro-F1 kosong = paper tidak mengutip angka itu di badan teks yang saya baca.')
print('Jangan diisi angka karangan.')

,Model,Macro-F1,Bisa di Colab?
0,ChatGPT-4,0.8450,tidak (API berbayar)
1,Human annotators,0.8460,ya (label asli)
2,LLaMA3.3_70B,0.8460,tidak (~140 GB)
3,LLaMA3.1_70B,NaN,tidak (~140 GB)
4,Gemma2_27B,NaN,tidak (~54 GB)
5,Qwen/Qwen2.5-3B-Instruct,0.5778,ya



Kolom Macro-F1 kosong = paper tidak mengutip angka itu di badan teks yang saya baca.
Jangan diisi angka karangan.


## 7. Laporan Akhir

### Ringkasan paper
Membandingkan 22 LLM open-source + ChatGPT-4 + anotator manusia pada dua tugas zero-shot untuk tweet Indonesia: sentimen 3 kelas (IndoNLU SmSA, 12.760 teks) dan emosi 5 kelas (IndoNLU EmoT, 4.401 tweet). Metrik: precision, recall, F1 per kelas + macro-F1, plus waktu inferensi. Hasil: ChatGPT-4 macro-F1 0,845, manusia 0,846, LLaMA 70B terbaik di antara open-source. Dijalankan di 4× RTX A6000 dengan Ollama.

### Sudah teruji vs belum

| | |
|---|---|
| **Sudah dijalankan & diverifikasi** | Unduh SmSA (12.760) + EmoT (4.401), keduanya cocok persis angka paper; parser label; konstruksi prompt |
| **Belum diuji** | Inferensi LLM zero-shot (`RUN_INFERENCE=False`), seluruh 22 model, ChatGPT-4 |

### [ASUMSI]
1. `SEED = 42` — paper tidak menyebut
2. `temperature=0`, `max_new_tokens=8` — paper tidak menyebut setelan sampling
3. Parser label: ambil kemunculan label pertama; fallback ke kelas mayoritas
4. Emosi diperlakukan **single-label** walau prompt paper membolehkan multi-label — dataset EmoT sendiri hanya punya satu label per tweet
5. Model uji = Qwen2.5-3B-Instruct (paper tidak menguji versi ini)

### Ancaman validitas
- **Tidak ada Data Availability statement** sama sekali di paper
- **Tidak ada kode.** Prompt ada, tapi setelan sampling, parsing output, dan penanganan jawaban non-label tidak dijelaskan
- **Hardware tidak terjangkau**: 4× RTX A6000. Hampir tidak ada peneliti Indonesia yang bisa mengulang benchmark 70B
- Paper menyebut LLaMA3.3 sebagai *"hypothetical"* dan *"we assume"* — tanda penulis tidak memverifikasi model yang dijalankan. Ini melemahkan seluruh tabel hasil
- Klaim "slightly outperforming human annotators" (0,845 vs 0,846) sebenarnya **tidak** mengungguli — 0,845 < 0,846. Paper menulis "slightly outperforming" padahal angkanya lebih rendah
